#**Step-1: Import the Libraries**

In [ ]:
import pandas as pd
import random, string

#**Step-2: Import Excel Sheet into Environment**

In [ ]:
data = pd.read_excel("/content/Dataset.xlsx", sheet_name="dataset")

#**Step-3: Drop Unnamed Column**

In [ ]:
data = data.drop('Unnamed: 0', axis=1)

#**Step-4: Capitalize all Columns of the DataFrame**

In [ ]:
Columns = [c.capitalize() for c in [i for i in data.columns]]

In [ ]:
data.columns = Columns

#**Step-5: Dealing with Missing Values**

a) Make a list of columns which have missing values in them.

In [ ]:
missing_value_columns = [c for c in Columns if data[c].isnull().any()]

b) Make a list of row indices which have missing values in them.

In [ ]:
rows_to_be_removed = [r for r in data[data[missing_value_columns].isnull().any(axis=1)].index]

c) Check the percentage of rows to be removed.

In [ ]:
total_rows = data.shape[0]
rows_with_nan = data[data.isnull().any(axis=1)].shape[0]
rows_without_nan = total_rows - rows_with_nan
percentage_nan_rows = (rows_with_nan / total_rows) * 100
print("Percentage of rows with NaN values:", percentage_nan_rows)

Percentage of rows with NaN values: 0.05087719298245614


d) Since it is less than 10%of the entire dataset, we will remove the rows.

In [ ]:
data = data.drop(rows_to_be_removed, axis=0)

#**Step-6: Dealing with Duplicate Values**

###**Problem:**

Since each track can belong to multiple genre, every time the genre changes the entire row of data is duplicated for each of its multiple genres

### **Pre-Processing:**

a) Create a dictionary of track ids with multiple genres like

{'Track-ID':['All Index Numbers which are duplicated']}

In [ ]:
duplicated_values_df = pd.DataFrame()
duplicate_track_ids = data['Track_id'][data['Track_id'].duplicated(keep=False)]
track_id_indices = {}
for id in duplicate_track_ids.unique():
    duplicate_rows_for_id = data[data['Track_id'] == id]
    duplicated_values_df = pd.concat([duplicated_values_df, duplicate_rows_for_id])
    track_id_indices[id] = duplicate_rows_for_id.index.tolist()
track_id_indices

b) Create a column called 'Genre Count', where Tracks with more than one genre are labelled 'Multiple-Genre' and Tracks with one genre is labelled as 'Single-Genre' for easy and better identification.

In [ ]:
duplicates_index_list = list(track_id_indices.values())

In [ ]:
multi_genre_indices = [item for sublist in duplicates_index_list for item in sublist]

In [ ]:
data['Genre_count'] = 'single genre'

In [ ]:
data.loc[multi_genre_indices, 'Genre_count'] = 'multi-genre'

###**Solution:**

#####**Part A**

Create a unique Genre-ID & Artist-ID for each duplicated track and shift them to a new Dataframe/Table for easy database schema construction

In [ ]:
# Make a list of Genre-IDs for track with multiple genre
multi_genre_id_list=[]
for i in range(len(track_id_indices)):
  multi_genre_id_list.append(''.join(random.choices(string.ascii_uppercase + string.ascii_lowercase + string.digits, k=7)))
multi_genre_id_list

In [ ]:
#Making a dictionary {'Genre-ID':[List of respective Indices where they need to be updated]}
multi_genre_ids = dict(zip(multi_genre_id_list, track_id_indices.values()))

In [ ]:
# Making Genre-IDs for entire dataframe irrespective of Genre Count
genre_id_list=[]
for i in range(data.shape[0]):
  genre_id_list.append(''.join(random.choices(string.ascii_uppercase + string.ascii_lowercase + string.digits, k=7)))

In [ ]:
#Create the column and add the values
data['Genre_id'] = genre_id_list

In [ ]:
#Change the Genre-IDs for their respective index
for i in list(multi_genre_ids.keys()):
    for j in multi_genre_ids[i]:
      data.loc[j, 'Genre_id'] = i

In [ ]:
# Make a list of Artist-IDs for track with multiple genre
artist_id_list=[]
for i in range(len(artist_names_list)):
  artist_id_list.append(''.join(random.choices(string.ascii_uppercase + string.digits, k=8)))
artist_id_list

In [ ]:
#Get unique artist names
artist_names_list= data[data.duplicated(subset=['Artists'])]['Artists'].unique()

In [ ]:
if len(artist_names_list) == len(artist_id_list):
    print("The two lists have the same length.")
else:
    print("The two lists have different lengths.")

The two lists have the same length.


In [ ]:
#Making a dictionary {'Artist-Name':'Artist-ID'}
artist_ids = dict(zip(artist_names_list, artist_id_list))

In [ ]:
#Making Artist-ID Column
data['Artist_id'] = data['Artists'].map(artist_ids).fillna(''.join(random.choices(string.ascii_uppercase + string.digits, k=8)))

#####**Part B**

Copy the dataframe to a new dataframe

In [ ]:
main_data = data.copy()

Copy the Genre-ID and Type Columns into another DataFrame - Genre.

In [ ]:
Genre = main_data[['Genre_id','Track_genre']]

Drop Track-Genre column from the main_data

In [ ]:
main_data.drop('Track_genre', axis=1)

#####**Part C**

Remove all the duplicate rows

In [ ]:
final_data = main_data.copy()

In [ ]:
indices_to_remove = []

In [ ]:
for id, indices in track_id_indices.items():
    indices_to_remove.extend(indices[1:])

In [ ]:
final_data = final_data.drop(indices_to_remove, axis=0)

#####**Part D**

Check if there are any more duplicates left.

In [ ]:
duplicate_track_ids_final = final_data['Track_id'][final_data['Track_id'].duplicated(keep=False)]

In [ ]:
if not duplicate_track_ids_final.empty:
    print("Duplicate track IDs found in 'example':")
    print(duplicate_track_ids_final.unique())
else:
    print("No duplicate track IDs found in 'example'.")

No duplicate track IDs found in 'example'.


In [ ]:
final_data = final_data.reset_index(drop=True).set_index(final_data.index + 1)

###**Summary**

The final dataframe has clean data.

The number of rows which were duplicated were 24,244 which rounds to about 21.28% of the total data.

By this method, we deal with duplicate values without loosing the information


#**Step-7: Making Dataframes to Export**

Creating the dataframes

In [ ]:
#Create a DataFrame of Artists
Artists = final_data[['Artist_id','Artists']]
#Remove Duplicate Values
Artists = Artists.drop_duplicates()

In [ ]:
#Assigning unique Attribute-ID to each track
final_data['Attribute_id'] = [''.join(random.choices(string.ascii_uppercase + string.digits, k=5)) for _ in range(len(final_data))]
# Assign the list to the 'Attribute_id' column
Attributes = final_data[['Attribute_id', 'Popularity', 'Duration_ms', 'Explicit', 'Danceability', 'Energy', 'Key', 'Loudness', 'Mode', 'Speechiness', 'Acousticness', 'Instrumentalness', 'Liveness', 'Valence', 'Tempo', 'Time_signature']]

In [ ]:
Tracks = final_data[['Track_id','Artist_id', 'Genre_id', 'Attribute_id', 'Track_name']]

#**Step-8: Exporting Dataframes to MySQL**

###**a) Installing Libraries:**

In [ ]:
pip install mysql-connector-python

###**b) Importing the Libraries**

In [ ]:
import mysql.connector

###**c) Establish MySQL Connection**

In [ ]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",     # Replace with your MySQL username
    password="root"  # Replace with your MySQL password
)

In [ ]:
cursor = conn.cursor()

###**d) Create Database and Tables**

Creating the table if it doesn't exist

In [ ]:
cursor.execute("CREATE DATABASE IF NOT EXISTS Spotify")
cursor.execute("USE Spotify")

Creating Artist Table

In [ ]:
cursor.execute("CREATE TABLE IF NOT EXISTS Artists_DET (Artist_id VARCHAR(8) PRIMARY KEY,Artist VARCHAR(255))")

Creating Genre Table

In [ ]:
cursor.excute("CREATE TABLE IF NOT EXISTS Genre_DET (Genre_id VARCHAR(7) PRIMARY KEY,Genre VARCHAR(255))")

Creating Attributes Table

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS Attributes_DET (
    Attribute_id VARCHAR(5) PRIMARY KEY,
    Popularity INT,
    Duration_ms INT,
    Explicit BOOLEAN,
    Danceability FLOAT,
    Energy FLOAT,
    Key INT,
    Loudness FLOAT,
    Mode INT,
    Speechiness FLOAT,
    Acousticness FLOAT,
    Instrumentalness FLOAT,
    Liveness FLOAT,
    Valence FLOAT,
    Tempo FLOAT,
    Time_signature INT
)
""")

Creating Tracks Table

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS Tracks_DET (
    Track_id VARCHAR(255) PRIMARY KEY,
    Artist_id VARCHAR(8),
    Genre_id VARCHAR(7),
    Attribute_id VARCHAR(5),
    Track_name VARCHAR(255),
    FOREIGN KEY (Artist_id) REFERENCES Artists_DET(Artist_id),
    FOREIGN KEY (Genre_id) REFERENCES Genre_DET(Genre_id),
    FOREIGN KEY (Attribute_id) REFERENCES Attributes_DET(Attribute_id)
)
""")

###**e) Inserting Data into MySQL**

Creating a dictionary of dataframes and their corresponding table names

In [ ]:
dataframe_table_map = {
    'Artists': 'Artists_DET',
    'Genre': 'Genre_DET',
    'Attributes': 'Attributes_DET',
    'Tracks': 'Tracks_DET'
}

Using for loop to export data of the dataframe to MySQL table

In [ ]:
for df_name, t_name in dataframe_table_map.items():
    df = locals()[df_name]
    for _, row in df.iterrows
    sql = f"INSERT INTO {table_name} ({', '.join(df.columns)}) VALUES ({', '.join(['%s'] * len(row))})"
    cursor.execute(sql, tuple(row))
conn.commit()

###**f) Closing the connection**

In [ ]:
cursor.close()
conn.close()